# 01 - Games table: validation and EDA

First look at `STEAM_DB.STEAM_RAW.GAMES`, loaded by dlt from the Steam Store `appdetails` endpoint. This notebook holds the validation SQL, what we saw, and what it means for dbt staging, so we do not need a separate `.sql` file and findings document.

**Scope**
- Source: one load on 2026-10-23 (about 14:09 Stockholm time), load id `1792757384.0749068`, 81 Paradox Interactive games.
- Portal 2 (`steam_appid = 620`) is an earlier test row and is excluded with `steam_appid <> 620`.
- Read-only: only `SELECT` and `DESCRIBE`. Nothing in Snowflake is changed.

**How to read it**
- The code cells are the queries. The text under each one (**What we saw**) was written by hand from the results on 2026-10-23. It is not generated, so the numbers will change after more loads.
- Queries marked **not yet run** were written after the first round of checks and still need to be run once.

**Limits:** one snapshot only, so there is no price or discount history yet. Steam gives no sales figures, so the review and player data we add later are proxies.

## 0. Connect to Snowflake

The next cells read the Snowflake settings from `.env` (same variable names as `.env.example`) and connect as the loader role `STEAM_LOADER`, which owns `STEAM_RAW`. No value is printed. Timestamps are shown in UTC (the `TIMEZONE` session parameter), so they do not depend on the account default.

Select the project's `.venv` as the notebook kernel.

In [1]:
import os
from pathlib import Path

import pandas as pd
import snowflake.connector

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [2]:
def find_repo_root() -> Path:
    """Walk up from the notebook's folder until we find pyproject.toml."""
    here = Path.cwd().resolve()
    return next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())


def load_env_file(path: Path) -> None:
    """Copy KEY=value lines from a .env file into os.environ (never prints values)."""
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


load_env_file(find_repo_root() / ".env")

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_LOADER_USER"],
    password=os.environ["SNOWFLAKE_LOADER_PASSWORD"],
    role=os.environ["SNOWFLAKE_LOADER_ROLE"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema="STEAM_RAW",
    session_parameters={"TIMEZONE": "UTC"},
)


def query(sql: str) -> pd.DataFrame:
    """Run one read-only statement and return the result as a DataFrame."""
    with conn.cursor() as cursor:
        cursor.execute(sql)
        columns = [column[0] for column in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=columns)

In [3]:
# Sanity check: we should be the loader role, in STEAM_DB.STEAM_RAW.
query("""
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_DATABASE() AS database, CURRENT_SCHEMA() AS schema
""")

,ROLE,WAREHOUSE,DATABASE,SCHEMA
0,STEAM_LOADER,STEAM_WH,STEAM_DB,STEAM_RAW


## 1. Rows per load

One row per game per collection run, so every load shows up as one `collected_at` value.

**What we saw:** the Paradox load has 81 rows (confirmed in section 6: `COUNT(*) = 81` with Portal 2 excluded). The raw output of this exact query was not saved. Expect two lines: Portal 2 (1 row) and the Paradox load (81 rows, 81 distinct games).

**Rule for later runs:** `append` adds a new set of rows on every run. Run the full load once per day and never just for testing.

In [4]:
query("""
SELECT collected_at, COUNT(*) AS rows_loaded, COUNT(DISTINCT steam_appid) AS distinct_games
FROM games
GROUP BY collected_at
ORDER BY collected_at
""")

,COLLECTED_AT,ROWS_LOADED,DISTINCT_GAMES
0,2026-10-23 09:21:10.061489+00:00,1,1
1,2026-10-23 12:09:44.091552+00:00,81,81


## 2. One line per game

Groups by game instead of by run. Every game should show `times_collected = 1` for now. Once we collect daily, this is where the price history builds up.

**Not yet run.** Expected: 82 rows (81 Paradox games plus Portal 2), each with `times_collected = 1`.

In [5]:
query("""
SELECT
    name,
    steam_appid,
    COUNT(*) AS times_collected,
    MIN(collected_at) AS first_collected,
    MAX(collected_at) AS last_collected
FROM games
GROUP BY name, steam_appid
ORDER BY name
""")

,NAME,STEAM_APPID,TIMES_COLLECTED,FIRST_COLLECTED,LAST_COLLECTED
0,A Game of Dwarves,200370,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
1,Academia : School Simulator,672630,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
2,Across the Obelisk,1385380,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
3,Afterworld,4920230,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
4,Age of Wonders,61500,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
5,Age of Wonders 4,1669000,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
6,Age of Wonders II: The Wizard's Throne,61510,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
7,Age of Wonders III,226840,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
8,Age of Wonders Shadow Magic,61520,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00
9,Age of Wonders: Planetfall,718850,1,2026-10-23 12:09:44.091552+00:00,2026-10-23 12:09:44.091552+00:00


## 3. Are all the IDs Paradox games?

The IDs came from summaries of Steam's publisher search, not from our own checking, so we look for games whose `publishers` does not contain `Paradox`.

**What we saw:** one row, *The Kings' Crusade* (`42920`), with `publishers = []`. Steam's data lists no publisher for it. The ID is most likely right (it came from the Paradox publisher search), but we have not checked it another way, and the data is incomplete. Every other game lists `Paradox Interactive`; some also list `Paradox Arc`.

**Decision for dbt:** the project scope is defined by the `GAME_APPIDS` list, not by the publisher field.

In [6]:
query("""
SELECT steam_appid, name, type, publishers
FROM games
WHERE steam_appid <> 620
  AND NOT CONTAINS(publishers::STRING, 'Paradox')
""")

,STEAM_APPID,NAME,TYPE,PUBLISHERS
0,42920,The Kings' Crusade,game,[]


## 4. Do the names match our key?

We compared 12 well-known IDs with the names in our notes: Crusader Kings III, Hearts of Iron IV, Stellaris, Cities: Skylines II, Victoria 3, Age of Wonders 4, Europa Universalis IV, Europa Universalis V, Cities: Skylines, Prison Architect, Vampire: The Masquerade - Bloodlines 2, Prison Architect 2.

**What we saw:** all 12 match, and all 12 have `type = game`. The Vampire title carries the symbols `(R)` and `(TM)` in its Steam name. The other 69 names were not compared one by one; section 3 covers them through the publisher check.

In [7]:
query("""
SELECT steam_appid, name, type
FROM games
WHERE steam_appid IN (
    1158310, 394360, 281990, 949230, 529340, 1669000,
    236850, 3450310, 255710, 233450, 532790, 1257610
)
ORDER BY steam_appid
""")

,STEAM_APPID,NAME,TYPE
0,233450,Prison Architect,game
1,236850,Europa Universalis IV,game
2,255710,Cities: Skylines,game
3,281990,Stellaris,game
4,394360,Hearts of Iron IV,game
5,529340,Victoria 3,game
6,532790,Vampire: The Masquerade® - Bloodlines™ 2,game
7,949230,Cities: Skylines II,game
8,1158310,Crusader Kings III,game
9,1257610,Prison Architect 2,game


## 5. Column types

**What we saw**
- `COLLECTED_AT` is `TIMESTAMP_LTZ(9)` (timestamp with local time zone). It is displayed in the session time zone: Snowsight showed Pacific time (`-0700`), the same moment as 14:09 in Sweden.
- `DEVELOPERS`, `PUBLISHERS`, `GENRES`, `CATEGORIES` are `VARIANT` (JSON lists), as intended.
- `STEAM_APPID` and all price, score and count columns are `NUMBER(19,0)`. Prices are whole numbers in minor units (4999 = 49.99 EUR).
- Booleans (`IS_FREE`, `COMING_SOON`, `PLATFORM_*`) are `BOOLEAN`; text columns are `VARCHAR`.
- dlt added `_DLT_LOAD_ID` and `_DLT_ID`. Every other column is nullable.

In [8]:
query("DESCRIBE TABLE games")

,name,type,kind,null?,default,primary key,unique key,check,expression,comment,policy name,privacy domain,write default
0,DEVELOPERS,VARIANT,COLUMN,Y,None,N,N,None,None,None,None,None,None
1,PUBLISHERS,VARIANT,COLUMN,Y,None,N,N,None,None,None,None,None,None
2,GENRES,VARIANT,COLUMN,Y,None,N,N,None,None,None,None,None,None
3,CATEGORIES,VARIANT,COLUMN,Y,None,N,N,None,None,None,None,None,None
4,STEAM_APPID,"NUMBER(19,0)",COLUMN,Y,None,N,N,None,None,None,None,None,None
5,COLLECTED_AT,TIMESTAMP_LTZ(9),COLUMN,Y,None,N,N,None,None,None,None,None,None
6,NAME,VARCHAR(16777216),COLUMN,Y,None,N,N,None,None,None,None,None,None
7,TYPE,VARCHAR(16777216),COLUMN,Y,None,N,N,None,None,None,None,None,None
8,IS_FREE,BOOLEAN,COLUMN,Y,None,N,N,None,None,None,None,None,None
9,REQUIRED_AGE,"NUMBER(19,0)",COLUMN,Y,None,N,N,None,None,None,None,None,None


## 6. Nulls, free games, unreleased games

**What we saw** (81 games):

| Measure | Count |
|---|---|
| games | 81 |
| no name | 0 |
| free games | 2 |
| coming soon (unreleased) | 4 |
| no price | 8 |
| no price, not free, not unreleased | 2 |
| no release date text | 0 |
| no Metacritic score | 36 |
| distinct currencies | 1 (EUR) |

The 8 games without a price are the 2 free games, the 4 unreleased games and 2 more. We read the last number as the 2 games listed in section 8. The column was named `NO_PRICE_EXPLAINED` in the exported file but the formula in this notebook is the unexplained one, so section 8 confirms it.

In [9]:
query("""
SELECT
    COUNT(*) AS games,
    COUNT_IF(name IS NULL) AS no_name,
    COUNT_IF(is_free) AS free_games,
    COUNT_IF(coming_soon) AS coming_soon,
    COUNT_IF(price_final IS NULL) AS no_price,
    COUNT_IF(price_final IS NULL AND NOT is_free AND NOT coming_soon) AS no_price_unexplained,
    COUNT_IF(release_date_text IS NULL OR release_date_text = '') AS no_release_date,
    COUNT_IF(metacritic_score IS NULL) AS no_metacritic,
    COUNT(DISTINCT price_currency) AS currencies
FROM games
WHERE steam_appid <> 620
""")

,GAMES,NO_NAME,FREE_GAMES,COMING_SOON,NO_PRICE,NO_PRICE_UNEXPLAINED,NO_RELEASE_DATE,NO_METACRITIC,CURRENCIES
0,81,0,2,4,8,2,0,36,1


## 7. The unreleased games

**What we saw:** Steam answered `success = true` for all four, with data but no price.

| steam_appid | name | release_date_text |
|---|---|---|
| 1257610 | Prison Architect 2 | To be announced |
| 4920230 | Afterworld | To be announced |
| 1579420 | Guardians of the Wild Sky | Coming soon |
| 1573550 | LEGO Skylines | Coming soon |

`coming_soon = TRUE`, `is_free = FALSE`, `price_final` is empty, and genres are filled in. The release text is not a date.

In [10]:
query("""
SELECT steam_appid, name, release_date_text, coming_soon, is_free, price_final, genres
FROM games
WHERE coming_soon
""")

,STEAM_APPID,NAME,RELEASE_DATE_TEXT,COMING_SOON,IS_FREE,PRICE_FINAL,GENRES
0,1257610,Prison Architect 2,To be announced,True,False,None,"[\n ""Simulation"",\n ""Strategy""\n]"
1,4920230,Afterworld,To be announced,True,False,None,"[\n ""Strategy""\n]"
2,1579420,Guardians of the Wild Sky,Coming soon,True,False,None,"[\n ""Action"",\n ""Adventure"",\n ""Indie"",\n ""Massively Multiplayer"",\n ""R..."
3,1573550,LEGO® Skylines,Coming soon,True,False,None,"[\n ""Simulation""\n]"


## 8. Games with no price that are neither free nor unreleased

**Not yet run.** This lists the 2 games counted in section 6. Possible reasons to check: delisted, sold only as part of a bundle, or region-restricted in Sweden.

In [11]:
query("""
SELECT steam_appid, name, release_date_text, is_free, coming_soon, price_final
FROM games
WHERE steam_appid <> 620
  AND price_final IS NULL
  AND NOT is_free
  AND NOT coming_soon
""")

,STEAM_APPID,NAME,RELEASE_DATE_TEXT,IS_FREE,COMING_SOON,PRICE_FINAL
0,73020,Majesty 2 Collection,"19 Apr, 2011",False,False,None
1,349790,Galaxy of Pen & Paper +1,"27 Jul, 2017",False,False,None


## 9. Are all rows games?

`appdetails` also returns DLC, software and more. Our scope is games.

**Not yet run.** Expected: one line, `game`, with 81 rows and 81 distinct games. Only the 12 games in section 4 have been confirmed as `game`.

In [12]:
query("""
SELECT type, COUNT(*) AS row_count, COUNT(DISTINCT steam_appid) AS distinct_games
FROM games
WHERE steam_appid <> 620
GROUP BY type
""")

,TYPE,ROW_COUNT,DISTINCT_GAMES
0,game,81,81


## 10. How common are the messy-data cases?

In the first full-table export we saw HTML entities in descriptions, repeated category names, several publishers or developers on one game, and empty websites. This query counts them.

**Not yet run, and untested SQL.** `ARRAY_SIZE` and `ARRAY_DISTINCT` on a `VARIANT` column should work, but we have not confirmed it. If Snowflake complains, paste the error and we fix the query.

In [13]:
query("""
SELECT
    COUNT_IF(short_description ILIKE ANY ('%&quot;%', '%&amp;%', '%&#%')) AS with_html_entities,
    COUNT_IF(website IS NULL OR website = '') AS no_website,
    COUNT_IF(recommendations_total IS NULL) AS no_recommendations,
    COUNT_IF(ARRAY_SIZE(publishers) > 1) AS multiple_publishers,
    COUNT_IF(ARRAY_SIZE(developers) > 1) AS multiple_developers,
    COUNT_IF(ARRAY_SIZE(categories) <> ARRAY_SIZE(ARRAY_DISTINCT(categories))) AS duplicate_categories,
    COUNT_IF(discount_percent > 0) AS discounted_now
FROM games
WHERE steam_appid <> 620
""")

,WITH_HTML_ENTITIES,NO_WEBSITE,NO_RECOMMENDATIONS,MULTIPLE_PUBLISHERS,MULTIPLE_DEVELOPERS,DUPLICATE_CATEGORIES,DISCOUNTED_NOW
0,3,13,11,3,4,2,5


## 11. Findings and decisions for dbt staging

Each finding ends with what staging should do. These are proposals until we build `feat/dbt-staging`.

1. **Grain.** One row per `steam_appid` per `collected_at`. Portal 2 (620) is a test row and is left out. The scope is the `GAME_APPIDS` list, because *The Kings' Crusade* has an empty `publishers` list (section 3).
   - *Staging:* filter on the project's game list (a seed or a source list), not on the publisher field.

2. **Prices.** Whole numbers in minor units, one currency (EUR) for `cc=se`. Prices are empty for free games (for example Crusader Kings II, now free to play), for the 4 unreleased games and for 2 more games to be explained (section 8). Discounts appear as `price_initial` > `price_final` with `discount_percent`; at load time, for example, Victoria 3 was 70 % off and Empire of Sin 75 % off.
   - *Staging:* `price / 100` as decimals, keep NULL for no price, keep `is_free` and `coming_soon` as separate flags so a missing price is never read as 0.

3. **Release date.** Free text such as `18 Apr, 2011`, but also `To be announced` and `Coming soon`.
   - *Staging:* try to parse the date (`TRY_TO_DATE` with a fixed format, to be tested) and get NULL for non-dates; keep the original text in its own column.

4. **Time.** `collected_at` is `TIMESTAMP_LTZ`, so it shows in whatever time zone the session uses.
   - *Staging:* convert explicitly (UTC, and a Stockholm date if we need a calendar day).

5. **Lists.** `developers`, `publishers`, `genres`, `categories` are JSON lists in `VARIANT`. Some games have several publishers (for example `Paradox Interactive` and `Paradox Arc`). Categories can repeat a value (for example `Steam Workshop`) and one raw untranslated key was seen (`#category_playable_at_your_own_pace`).
   - *Staging:* unpack with `LATERAL FLATTEN` into one row per game and value, remove duplicates, and decide what to do with raw keys.

6. **Text.** `short_description` has HTML entities such as `&quot;` and `&amp;`.
   - *Staging:* decode or replace them in one place (a macro), not in every model.

7. **Missing scores.** 36 of 81 games (44 %) have no Metacritic score, so it cannot be a main metric. Review sentiment from `appreviews` and player counts have to carry the reception analysis.

8. **Unreleased games.** Four titles are kept on purpose. Their release text is not a date and they have no price. They become releases when `coming_soon` turns FALSE.

9. **History.** Only one collection exists, so there is no price history yet. Each extra load adds 81 rows; duplicates by `steam_appid` are expected and are the history, not an error.

## 12. Open items

- Run sections 2, 8, 9 and 10 and write down what we saw.
- Explain the 2 games with no price that are not free or unreleased.
- Compare all 81 names with Steam's publisher page, not only 12.
- Decide the rule for repeated runs (proposal: one scheduled run per day, and use the latest `collected_at` for current-state questions).
- Add a progress line to the load log (a full run is silent for about 3.5 minutes).
- Still open from the infrastructure docs: `DLT_USER` account type (PERSON vs SERVICE) and a non-interactive login before any scheduled run.

In [14]:
conn.close()